# Exploratory Data Analysis for the Course Recommender System

This notebook prepares the dataset understanding used by the capstone presentation: course topics, genre distribution, enrollment behaviour, and the most popular courses.

## One-time setup

Run this once in your terminal before opening the notebooks on your PC:

```bash
pip install -r requirements.txt
```

The notebook reads CSV files from the local `datasets` folder. If a file is missing, it downloads the official IBM Skills Network dataset automatically.

In [ ]:
from pathlib import Path
import urllib.request
import pandas as pd
import numpy as np

DATA_DIR = Path("datasets")
DATA_URLS = {
    "ratings.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/ratings.csv",
    "course_genre.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/course_genre.csv",
    "rs_content_test.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/rs_content_test.csv",
    "user_profile.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/user_profile.csv",
    "course_processed.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/course_processed.csv",
    "courses_bows.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/courses_bows.csv",
    "sim.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/sim.csv",
}

def ensure_dataset(filename):
    DATA_DIR.mkdir(exist_ok=True)
    path = DATA_DIR / filename
    if not path.exists():
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(DATA_URLS[filename], path)
    return path

def load_csv(filename, **kwargs):
    return pd.read_csv(ensure_dataset(filename), **kwargs)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)

In [ ]:
course_df = load_csv("course_genre.csv")
ratings_df = load_csv("ratings.csv")

genre_cols = [col for col in course_df.columns if col not in ["COURSE_ID", "TITLE"]]

print("Course genre data:", course_df.shape)
print("Ratings data:", ratings_df.shape)
print("Unique courses in metadata:", course_df["COURSE_ID"].nunique())
print("Unique users in ratings:", ratings_df["user"].nunique())
print("Unique rated courses:", ratings_df["item"].nunique())

course_df.head()

## Course Title Keywords

The presentation highlights recurring IT skill themes such as Python, Data Science, Machine Learning, Big Data, AI, TensorFlow, Containers, and Cloud Computing. The code below creates a word cloud when `wordcloud` is installed; otherwise it prints the most common title words.

In [ ]:
import re
from collections import Counter

title_text = " ".join(course_df["TITLE"].fillna("").astype(str)).lower()
tokens = re.findall(r"[a-zA-Z][a-zA-Z]+", title_text)
stop_words = {"with", "and", "for", "the", "introduction", "using", "your", "data", "science"}
word_counts = Counter(token for token in tokens if token not in stop_words)

try:
    import matplotlib.pyplot as plt
    from wordcloud import WordCloud

    wc = WordCloud(width=1000, height=450, background_color="white", colormap="viridis").generate(title_text)
    plt.figure(figsize=(14, 6))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title("Frequent Keywords in Course Titles")
    plt.show()
except ImportError:
    print("Install wordcloud and matplotlib to display the word cloud.")
    print(word_counts.most_common(20))

## Course Genre Distribution

Genre counts show what types of content are most represented on the platform. These genre vectors are later reused for content-based recommendation.

In [ ]:
genre_counts = (
    course_df[genre_cols]
    .sum()
    .sort_values(ascending=False)
    .rename("course_count")
    .reset_index()
    .rename(columns={"index": "genre"})
)

display(genre_counts)

try:
    import matplotlib.pyplot as plt
    import seaborn as sns

    plt.figure(figsize=(12, 5))
    sns.barplot(data=genre_counts, x="genre", y="course_count", color="#4C78A8")
    plt.xticks(rotation=45, ha="right")
    plt.title("Course Count by Genre")
    plt.xlabel("Genre")
    plt.ylabel("Number of Courses")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("Install matplotlib and seaborn to display the chart.")

## Enrollment Behaviour

The ratings file records course interactions. In this dataset, rating `2.0` means the user audited the course and `3.0` means the user completed it or earned a certificate.

In [ ]:
user_activity = ratings_df.groupby("user").size().rename("rating_count")
rating_distribution = ratings_df["rating"].value_counts().sort_index().rename("count")

print("Total rating/enrollment records:", len(ratings_df))
print("Unique users:", ratings_df["user"].nunique())
print("Unique courses:", ratings_df["item"].nunique())
display(rating_distribution)
display(user_activity.describe())

try:
    import matplotlib.pyplot as plt

    plt.figure(figsize=(10, 5))
    user_activity.clip(upper=user_activity.quantile(0.99)).hist(bins=40, color="#59A14F")
    plt.title("Distribution of User Rating Counts")
    plt.xlabel("Ratings per User, clipped at 99th percentile")
    plt.ylabel("Number of Users")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("Install matplotlib to display the histogram.")

## Top 20 Most Popular Courses

The most popular courses provide a popularity baseline and show the concentration of learner demand.

In [ ]:
top_courses = (
    ratings_df.groupby("item")
    .size()
    .sort_values(ascending=False)
    .head(20)
    .rename("enrollment_count")
    .reset_index()
    .rename(columns={"item": "COURSE_ID"})
    .merge(course_df[["COURSE_ID", "TITLE"]], on="COURSE_ID", how="left")
)

top20_share = top_courses["enrollment_count"].sum() / len(ratings_df) * 100

display(top_courses)
print(f"Top 20 courses account for {top20_share:.1f}% of all enrollments.")

try:
    import matplotlib.pyplot as plt
    import seaborn as sns

    plt.figure(figsize=(12, 7))
    sns.barplot(data=top_courses, y="TITLE", x="enrollment_count", color="#F28E2B")
    plt.title("Top 20 Courses by Enrollment Count")
    plt.xlabel("Enrollment Count")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("Install matplotlib and seaborn to display the chart.")

## EDA Takeaways

- The platform is strongly centered around Python, Data Science, Machine Learning, Big Data, Cloud, and related technical skills.
- Demand is concentrated: the top 20 courses account for about 63.3% of all enrollments.
- User activity is long-tailed, which means collaborative filtering must handle sparse user-course interactions.
- The 14 course genre columns provide an interpretable feature space for content-based recommendation.